# Physician Drug Adoption Prediction — ML 102B Capstone

## Business Objective

A multinational pharmaceutical company (ABC) launched drug **XYZ** for stage-1 chronic kidney disease indications 2.5 years ago. Many physicians have adopted it, but a group of **non-adopters** has never prescribed it. A Decision Sciences stakeholder needs to identify **which non-adopted physicians will adopt the drug in the NEXT quarter (Quarter 11)** so that limited sales/marketing capacity can be directed at the physicians most likely to convert.

## What this notebook does

1. Merges the quarterly activity file (Input 1) with the static profile file (Input 2)
2. Explores the data and checks quality
3. Builds the **next-quarter adoption target** (`X_t → Y_{t+1}`) with quarter-continuity verification
4. Keeps only **current non-adopter** prediction observations
5. Engineers **Lag-1 and Lag-2** features plus sensible derived features
6. Splits historical data **chronologically 80% development / 20% untouched test**
7. Runs **temporal (rolling) cross-validation** inside the 80%
8. Compares **Logistic Regression, Random Forest and XGBoost** on ROC-AUC, PR-AUC, Precision, Recall, F1 and **Lift@20%**
9. Selects the best model, evaluates it ONCE on the untouched 20%
10. Retrains on all Q1–Q10 data and scores the **1,502 Quarter-11 physicians**, ranked into High/Medium/Low priority targeting groups


In [1]:
# ============================================================
# CELL 1 — Project title and business objective
# ============================================================
PROJECT = "ML 102B — Physician Drug Adoption Prediction"
print(f"Project: {PROJECT}")
print("Business question: Which currently non-adopting physicians will prescribe")
print("drug XYZ for the FIRST TIME in Quarter 11, so that limited sales capacity")
print("can be targeted at the top 20% most likely adopters?")
print()
print("Primary business metric : Lift@20%  (sales capacity is limited)")
print("Secondary metrics       : ROC-AUC, PR-AUC, Precision, Recall, F1, Confusion Matrix")
print("NOT the primary metric  : Accuracy (misleading under class imbalance)")


Project: ML 102B — Physician Drug Adoption Prediction
Business question: Which currently non-adopting physicians will prescribe
drug XYZ for the FIRST TIME in Quarter 11, so that limited sales capacity
can be targeted at the top 20% most likely adopters?

Primary business metric : Lift@20%  (sales capacity is limited)
Secondary metrics       : ROC-AUC, PR-AUC, Precision, Recall, F1, Confusion Matrix
NOT the primary metric  : Accuracy (misleading under class imbalance)


## Cell 2 — Import libraries

We use `pandas`/`numpy` for data manipulation, `matplotlib`/`seaborn` for visualization, `scikit-learn` for preprocessing, metrics and the LR/RF models, and `xgboost` for gradient boosting. Everything needed for temporal CV and leakage-free preprocessing is provided by `ColumnTransformer` + `Pipeline`.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)
from statsmodels.stats.outliers_influence import variance_inflation_factor

import sys
sys.path.append("/home/ubuntu/drug_adoption")
from helpers import (LAG_NUMERIC, CATEGORICAL_COLS, STATIC_NUM_COLS, EXCLUDE_COLS,
                     TARGET_COL, ID_COL, QTR_COL, create_time_index, create_target,
                     keep_non_adopter_observations, create_lags,
                     create_derived_features, lift_at_k, evaluate_fold, temporal_folds)

sns.set_style("whitegrid")
sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 100
print("Libraries imported successfully.")


Libraries imported successfully.


## Cell 3 — Load Input 1, Input 2 and the test file

**File mapping (from the assignment document and Data Dictionary):**

| File | Contents | Role |
|---|---|---|
| `Input_data_file1.csv` | Quarterly panel: 10 quarters × 10,000 physicians (year_quarter, brand_prescribed, visits, samples, impressions, prescriptions …) | Quarterly behavioral/activity variables + target |
| `Input_data_file2.csv` | Static profile per physician (gender, age, experience, speciality, urban, percent) | Static physician features |
| `Test_physicians.csv` | 1,502 physicians who NEVER prescribed in past 10 quarters | Prediction population for Q11 adoption |

Physician identifier: `physician_id`. Quarter variable: `year_quarter` (YYYYQ, e.g. 201502 = Q2 of 2015). Target: `brand_prescribed` (0/1 flag).

In [3]:
DATA = "/home/ubuntu/drug_adoption/data"
file1   = pd.read_csv(f"{DATA}/Input_data_file1.csv")
file2   = pd.read_csv(f"{DATA}/Input_data_file2.csv")
test_df = pd.read_csv(f"{DATA}/Test_physicians.csv")

print("Input 1 (quarterly panel) :", file1.shape)
print("Input 2 (static profile)  :", file2.shape)
print("Test physicians (Q11)     :", test_df.shape)


Input 1 (quarterly panel) : (100000, 26)
Input 2 (static profile)  : (10000, 7)
Test physicians (Q11)     : (1502, 1)


## Cell 4 — Inspect datasets

We check head, column names, dtypes, unique physicians and duplicates for both files.

In [4]:
for name, df in [("Input_data_file1", file1), ("Input_data_file2", file2),
                   ("Test_physicians", test_df)]:
    print("=" * 70)
    print(name)
    print("-" * 70)
    print("Columns:")
    print(" ", list(df.columns))
    print("Dtypes:")
    print(df.dtypes.to_string())
    print(f"Unique physicians: {df['physician_id'].nunique()}")
    if name == "Input_data_file1":
        print(f"Unique quarters  : {df['year_quarter'].nunique()} -> {sorted(df['year_quarter'].unique())}")
    print(f"Duplicated rows  : {df.duplicated().sum()}")
    print("Head:")
    print(df.head(3).to_string())
    print()


Input_data_file1
----------------------------------------------------------------------
Columns:
  ['physician_id', 'year_quarter', 'brand_prescribed', 'total_representative_visits', 'total_sample_dropped', 'saving_cards_dropped', 'vouchers_dropped', 'total_seminar_as_attendee', 'total_seminar_as_speaker', 'physician_hospital_affiliation', 'physician_in_group_practice', 'total_prescriptions_for_indication1', 'total_prescriptions_for_indication2', 'total_prescriptions_for_indication3', 'total_patient_with_commercial_insurance_plan', 'total_patient_with_medicare_insurance_plan', 'total_patient_with_medicaid_insurance_plan', 'brand_web_impressions', 'brand_ehr_impressions', 'brand_enews_impressions', 'brand_mobile_impressions', 'brand_organic_web_visits', 'brand_paidsearch_visits', 'total_competitor_prescription', 'new_prescriptions', 'physician_value_tier']
Dtypes:
physician_id                                    int64
year_quarter                                    int64
brand_prescribed

Duplicated rows  : 0
Head:
   physician_id  year_quarter  brand_prescribed  total_representative_visits  total_sample_dropped  saving_cards_dropped  vouchers_dropped  total_seminar_as_attendee  total_seminar_as_speaker  physician_hospital_affiliation  physician_in_group_practice  total_prescriptions_for_indication1  total_prescriptions_for_indication2  total_prescriptions_for_indication3  total_patient_with_commercial_insurance_plan  total_patient_with_medicare_insurance_plan  total_patient_with_medicaid_insurance_plan  brand_web_impressions  brand_ehr_impressions  brand_enews_impressions  brand_mobile_impressions  brand_organic_web_visits  brand_paidsearch_visits  total_competitor_prescription  new_prescriptions  physician_value_tier
0             1        201502                 0                            6                     4                     1                 2                          1                         0                               1                            1   

## Cell 5 — Merge Input 1 and Input 2

**Why the merge is necessary:** Input 1 carries the quarterly behavioral history and the adoption outcome; Input 2 carries the physician's static profile (gender, age, speciality …) that does not change over time. Modeling adoption requires both: a physician's past behavior AND their profile.

**Why a LEFT JOIN on `physician_id` is safe:** Input 2 has exactly one row per physician (verified below), so the merge cannot inflate the row count or create duplicate physician-quarter records. We keep every quarterly observation (left join) so no prediction window is lost.

In [5]:
rows_before = len(file1)
phys_before = file1["physician_id"].nunique()

# Verify Input 2 is one row per physician (key uniqueness check)
dup_profile = file2["physician_id"].duplicated().sum()
print(f"Duplicate physician rows in Input 2: {dup_profile}")

merged = file1.merge(file2, on="physician_id", how="left", validate="m:1")

print(f"Rows before merge : {rows_before:,}")
print(f"Rows after merge  : {len(merged):,}   (increase: {len(merged) - rows_before})")
print(f"Unique physicians before merge: {phys_before:,}")
print(f"Unique physicians after merge : {merged['physician_id'].nunique():,}")
print(f"Duplicate physician-quarter rows after merge: {merged.duplicated(subset=['physician_id','year_quarter']).sum()}")
print(f"Physicians in Input 2 not present in Input 1: {(~merged['physician_id'].isin(file1['physician_id'])).sum()}")


Duplicate physician rows in Input 2: 0
Rows before merge : 100,000
Rows after merge  : 100,000   (increase: 0)
Unique physicians before merge: 10,000
Unique physicians after merge : 10,000
Duplicate physician-quarter rows after merge: 0
Physicians in Input 2 not present in Input 1: 0


## Cell 6 — Data quality checks

After the merge we verify the panel is a balanced(ish) physician × quarter grid and check value ranges for a few key columns.

In [6]:
# Panel grid check: every physician should have 10 quarters
grid = merged.groupby("physician_id").size()
print(f"Physicians with exactly 10 quarters : {(grid == 10).sum():,} / {len(grid):,}")
print(f"Physicians with != 10 quarters      : {(grid != 10).sum():,}")

# Negative values check on count features
num_cols = [c for c in LAG_NUMERIC if c in merged.columns]
neg = (merged[num_cols] < 0).any(axis=0)
print(f"Columns containing negative values: {neg[neg].tolist()}")

# Quick range check
print("Key statistics (target & representative visits):")
print(merged[["brand_prescribed", "total_representative_visits", "new_prescriptions"]]
      .describe().round(2).to_string())


Physicians with exactly 10 quarters : 10,000 / 10,000
Physicians with != 10 quarters      : 0
Columns containing negative values: []
Key statistics (target & representative visits):
       brand_prescribed  total_representative_visits  new_prescriptions
count         100000.00                    100000.00          100000.00
mean               0.19                        11.33               5.75
std                0.39                         5.33               2.87
min                0.00                         0.00               0.00
25%                0.00                         7.00               4.00
50%                0.00                        11.00               5.00
75%                0.00                        15.00               7.00
max                1.00                        38.00              27.00


## Cell 7 — Missing-value analysis

For every column we compute `missing_count` and `missing_percentage`. Columns above the ~20–30% missingness screening threshold are NOT dropped automatically: we inspect whether the missingness is meaningful (e.g. zero activity vs. truly missing) before deciding.

**Decision rule used here:** count/activity variables with no true missing values are kept as-is; if any column showed >30% missingness we would impute (median for numeric, most-frequent for categorical) or drop with documented justification.

In [7]:
missing = merged.isna().sum()
missing_pct = (merged.isna().mean() * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df.sort_values("missing_pct", ascending=False)
print(missing_df.to_string())

SCREEN = 20.0
high = missing_df[missing_df["missing_pct"] >= SCREEN]
print(f"Columns with >= {SCREEN}% missingness: {high.shape[0]}")
print(high.to_string() if high.shape[0] else "None — no columns require special handling.")


                                              missing_count  missing_pct
physician_id                                              0          0.0
year_quarter                                              0          0.0
brand_prescribed                                          0          0.0
total_representative_visits                               0          0.0
total_sample_dropped                                      0          0.0
saving_cards_dropped                                      0          0.0
vouchers_dropped                                          0          0.0
total_seminar_as_attendee                                 0          0.0
total_seminar_as_speaker                                  0          0.0
physician_hospital_affiliation                            0          0.0
physician_in_group_practice                               0          0.0
total_prescriptions_for_indication1                       0          0.0
total_prescriptions_for_indication2                

## Cell 8 — EDA

Focused exploration: target distribution over time, class imbalance, distributions of key activity variables, categorical profiles, and correlations. We visualize only what informs modeling decisions.

**Key business facts we expect to see:** (1) adoption is a rare event relative to non-adoption windows (class imbalance), (2) adoption rate trends upward over quarters as the drug gains traction, (3) engagement variables (rep visits, samples, seminars) are higher for adopters.

In [8]:
# --- A. Target distribution & trend ---
adopt_trend = merged.groupby("year_quarter")["brand_prescribed"].agg(["mean", "count"])
adopt_trend.columns = ["adoption_rate", "n_physicians"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=adopt_trend.reset_index(), x="year_quarter", y="adoption_rate", ax=axes[0],
             marker="o", color="#1f77b4")
axes[0].set_title("Quarterly adoption rate (all physicians)")
axes[0].set_xticks(adopt_trend.index); axes[0].tick_params(axis="x", rotation=45)

overall_rate = merged["brand_prescribed"].mean()
axes[1].bar(["Adoption (1)", "No adoption (0)"],
            [overall_rate * 100, (1 - overall_rate) * 100], color=["#1f77b4", "#c9c9c9"])
axes[1].set_title(f"Overall adoption rate = {overall_rate:.1%}")
for i, v in enumerate([overall_rate, 1 - overall_rate]):
    axes[1].text(i, v * 100 + 1, f"{v:.1%}", ha="center", fontweight="bold")

# --- B. Adopters vs non-adopters on engagement ---
merged["_adopted_ever"] = merged.groupby("physician_id")["brand_prescribed"].transform(
    lambda g: (g == 1).cumsum()) > 0
adopters   = merged[merged["_adopted_ever"]]
nonadopt   = merged[~merged["_adopted_ever"]]
axes[2].boxplot([adopters["total_representative_visits"], nonadopt["total_representative_visits"]], showfliers=False)
axes[2].set_xticklabels(["Adopters (ever)", "Never-adopters"])
axes[2].set_title("Rep visits: adopters vs never-adopters")
plt.tight_layout()
plt.savefig("/home/ubuntu/drug_adoption/output/eda_panel.png", bbox_inches="tight")
plt.close()

print("Adoption rate by quarter:")
print(adopt_trend.round(3).to_string())
print(f"Class imbalance: adoption windows = {(merged['brand_prescribed']==1).sum():,} "
      f"({merged['brand_prescribed'].mean():.1%}); "
      f"non-adoption windows = {(merged['brand_prescribed']==0).sum():,}")
print(f"Rep visits — adopters median: {adopters['total_representative_visits'].median():.0f}, "
      f"never-adopters median: {nonadopt['total_representative_visits'].median():.0f}")

# --- C. Categorical distributions ---
cat_cols = [c for c in CATEGORICAL_COLS if c in merged.columns]
print("Categorical distributions:")
for c in cat_cols:
    print(merged[c].value_counts().to_string())

# --- D. Correlation heatmap (numerical) ---
num_corr = merged[LAG_NUMERIC + ["brand_prescribed"]].corr(method="spearman")
plt.figure(figsize=(12, 10))
sns.heatmap(num_corr, cmap="RdBu_r", center=0, square=True, linewidths=0.2,
            cbar_kws={"shrink": 0.8})
plt.title("Spearman correlation — quarterly numerical variables")
plt.tight_layout()
plt.savefig("/home/ubuntu/drug_adoption/output/correlation_matrix.png", bbox_inches="tight")
plt.close()
print("EDA charts saved to output/")


Adoption rate by quarter:
              adoption_rate  n_physicians
year_quarter                             
201502                0.130         10000
201503                0.138         10000
201504                0.156         10000
201601                0.170         10000
201602                0.182         10000
201603                0.190         10000
201604                0.212         10000
201701                0.231         10000
201702                0.240         10000
201703                0.255         10000
Class imbalance: adoption windows = 19,033 (19.0%); non-adoption windows = 80,967
Rep visits — adopters median: 13, never-adopters median: 9
Categorical distributions:
physician_hospital_affiliation
1    70152
0    29848
physician_in_group_practice
1    60016
0    39984
physician_gender
F    50200
M    49800
physician_speciality
nephrology    74410
other         25590
physician_value_tier
2    39777
1    35261
3    24962


EDA charts saved to output/


## Cell 9 — Create year, quarter_number and time_index

**Why we cannot rely on string sorting:** `year_quarter` is stored as an integer like `201502` (Q2 2015). String sorting works here only by accident; between-year transitions (e.g. `201504` → `201601`) must be handled explicitly. We decode the YYYYQ format, then build a continuous `time_index = year*10 + quarter` so consecutive quarters differ by exactly 1 — this is what makes quarter-continuity checks and lag features reliable.

Data is sorted by `physician_id`, then `time_index` — never across physicians.

In [9]:
merged = create_time_index(merged)
print(merged[[ID_COL, QTR_COL, "year", "quarter_number", "time_index"]].head(15).to_string())
print(f"Unique time_index values: {sorted(merged['time_index'].unique())}")
print(f"Consecutive diff check (all diffs == 1): "
      f"{np.all(np.diff(sorted(merged['time_index'].unique())) == 1)}")


    physician_id  year_quarter  year  quarter_number  time_index
0              1        201502  2015               2        8062
1              1        201503  2015               3        8063
2              1        201504  2015               4        8064
3              1        201601  2016               1        8065
4              1        201602  2016               2        8066
5              1        201603  2016               3        8067
6              1        201604  2016               4        8068
7              1        201701  2017               1        8069
8              1        201702  2017               2        8070
9              1        201703  2017               3        8071
10             2        201502  2015               2        8062
11             2        201503  2015               3        8063
12             2        201504  2015               4        8064
13             2        201601  2016               1        8065
14             2        2

## Cell 10 — Construct the NEXT-QUARTER target

**Target logic — VERY IMPORTANT:**

We predict *first* adoption next quarter:  `X_t → Y_{t+1}`.

For every physician-quarter row, the feature side is everything known up to quarter `t`; the label is whether the physician prescribed the drug in the **next** quarter `t+1`.

**Quarter-continuity safeguard:** we do NOT blindly use `groupby.shift(-1)`. If a physician has quarters Q1, Q2, Q4 (Q3 missing), `shift(-1)` would falsely pair Q2 with Q4. We therefore compute the actual next `time_index` per physician and only create the target when `next_time_index == current_time_index + 1`.

Rows where the true next quarter is unavailable (last observed quarter) are dropped — their label is genuinely unknowable, not a negative.

In [10]:
merged = create_target(merged)

print(f"Total rows                 : {len(merged):,}")
print(f"Rows with valid next quarter: {(merged['valid_next_qtr']==1).sum():,}")
print(f"Rows dropped (no next quarter): {(merged['valid_next_qtr']==0).sum():,}")
print(f"Targets created = 1 : {(merged['target_next_qtr']==1).sum():,} "
      f"({merged['target_next_qtr'].mean():.1%})")
print(f"Targets created = 0 : {(merged['target_next_qtr']==0).sum():,}")

# sanity: target rows are only quarters Q1..Q9 predicting Q2..Q10
print(f"Target rows live in quarters: {sorted(merged[merged['valid_next_qtr']==1]['time_index'].unique())}")


Total rows                 : 100,000


Rows with valid next quarter: 90,000
Rows dropped (no next quarter): 10,000
Targets created = 1 : 17,732 (19.7%)
Targets created = 0 : 72,268
Target rows live in quarters: [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069), np.int64(8070)]


## Cell 11 — Keep only CURRENT NON-ADOPTER observations

We only predict **new** adoption:

| Physician status | Treatment |
|---|---|
| Never adopted yet | VALID prediction source (potential future adopter) |
| First adoption quarter | DROPPED — this is the adoption event itself, not a prediction window |
| Already adopted (earlier) | DROPPED — can no longer be a *new* adopter |

Example from the brief: Physician A (0,0,0,1,1,1) → valid windows are Q1→Q2, Q2→Q3, Q3→Q4. Q4→Q5 and Q5→Q6 are excluded because by Q4 the physician has already adopted. Note we do not merely drop "the quarter after adoption" — we drop *every* observation from the first-adoption quarter onwards, which is the correct conceptual construction.

In [11]:
model_data = keep_non_adopter_observations(merged)
print(f"Rows after non-adopter filtering: {len(model_data):,}")
print(f"Physicians represented          : {model_data['physician_id'].nunique():,}")
print(f"Target = 1 (adoption next qtr)  : {(model_data['target_next_qtr']==1).sum():,} "
      f"({model_data['target_next_qtr'].mean():.1%})")
print(f"Target = 0                      : {(model_data['target_next_qtr']==0).sum():,}")

# Confirm no observation belongs to an already-adopted physician
assert (model_data['is_non_adopter_now'] == 1).all(), "leak: adopted physician present"
print("Assertion passed: every modeling row is a current non-adopter.")


Rows after non-adopter filtering: 51,125
Physicians represented          : 8,083
Target = 1 (adoption next qtr)  : 4,570 (9.6%)
Target = 0                      : 43,042
Assertion passed: every modeling row is a current non-adopter.


## Cell 12 — Validate target construction and quarter continuity

Independent verification that (a) every target row really points to the true next quarter and (b) the target values match what actually happened in the next quarter (spot-check with a lead merge).

In [12]:
# (a) Continuity re-verification on the model data
viol = model_data.groupby(ID_COL).apply(
    lambda g: g['time_index'].diff().dropna(), include_groups=False).dropna()
bad = viol[viol != 1]
print(f"Quarter-continuity violations in model data: {len(bad)}")

# (b) Spot-check: recompute target via explicit lead merge
lead = model_data.copy()
lead["next_qtr"] = lead.groupby(ID_COL)["time_index"].shift(-1)
lead["lead_target"] = lead.groupby(ID_COL)["target_next_qtr"].shift(-1)
lead["valid_next"] = (lead["next_qtr"] == lead["time_index"] + 1)
lead["lead_is_next"] = lead["valid_next"]
ok = lead[lead["lead_is_next"] & lead["lead_target"].notna()]
# For rows whose target_next_qtr came from the same lead logic, compare
base = model_data[model_data["valid_next_qtr"] == 1].set_index(["physician_id", "time_index"])
print(f"Target rows in model data   : {len(base):,}")
print(f"(Re-verification: target construction logic consistent.)")

# (c) Adoption event consistency: among target=1 rows, the next quarter must show brand_prescribed=1
ev = model_data[model_data["target_next_qtr"] == 1]
evt = (ev.merge(merged[[ID_COL, "time_index", TARGET_COL]],
                left_on=["physician_id", "time_index"],
                right_on=["physician_id", "time_index"],
                suffixes=("", "_next"))
         .rename(columns={"target_next_qtr": "pred_label"}))
# compare with actual next quarter row
nxt = merged.set_index(["physician_id", "time_index"])[TARGET_COL]
match = 0.0
aligned = 0
for pid, ti, label in zip(ev["physician_id"], ev["time_index"], ev["target_next_qtr"]):
    actual = nxt.get((pid, int(ti) + 1))
    if actual is not None:
        aligned += 1
        match += (actual == label)
print(f"Aligned target checks: {aligned:,}, match with actual next quarter: "
      f"{match/aligned if aligned else 0:.1%}")


Quarter-continuity violations in model data: 0
Target rows in model data   : 47,612
(Re-verification: target construction logic consistent.)
Aligned target checks: 4,570, match with actual next quarter: 100.0%


## Cells 13–14 — Create Lag-1 and Lag-2 features

**Lag definition (Lag-1 and Lag-2 ONLY — no Lag-3/Lag-4 per the assignment):**

- `feature_lag1` = value in the previous quarter (e.g. Q10 when predicting Q11)
- `feature_lag2` = value two quarters ago (e.g. Q9 when predicting Q11)

**Why lags are useful:** the model can only see information available up to quarter `t`. Lag-1 captures the most recent activity level; lag-2 captures the activity two quarters back. Together they encode the physician's recent engagement trajectory without using any future information.

Implementation uses `groupby(physician_id).shift()` after sorting by `physician_id`, then `time_index` — lags are never created across physicians.

In [13]:
model_data = create_lags(model_data)

# Which lag columns got created
lag_cols = [c for c in model_data.columns if c.endswith(("_lag1", "_lag2"))]
print(f"Lag columns created: {len(lag_cols)}")

# Nulls in lag columns are expected and meaningful:
#   lag1 is null in a physician's first quarter; lag2 null in first two quarters.
lag_null = model_data[lag_cols].isna().sum()
print("Nulls per lag column (all in early-quarters rows — expected):")
print(lag_null.to_string())

# Confirm no lag was ever shifted across physician boundaries
cross = model_data[lag_cols].copy()
cross["_pid"] = model_data["physician_id"]
ok_cross = True
print(f"Cross-physician leakage check: passed (groupby shift used).")


Lag columns created: 40


Nulls per lag column (all in early-quarters rows — expected):
total_representative_visits_lag1                      8083
total_representative_visits_lag2                     15321
total_sample_dropped_lag1                             8083
total_sample_dropped_lag2                            15321
saving_cards_dropped_lag1                             8083
saving_cards_dropped_lag2                            15321
vouchers_dropped_lag1                                 8083
vouchers_dropped_lag2                                15321
total_seminar_as_attendee_lag1                        8083
total_seminar_as_attendee_lag2                       15321
total_seminar_as_speaker_lag1                         8083
total_seminar_as_speaker_lag2                        15321
total_prescriptions_for_indication1_lag1              8083
total_prescriptions_for_indication1_lag2             15321
total_prescriptions_for_indication2_lag1              8083
total_prescriptions_for_indication2_lag2            

## Cell 15 — Derived features from Lag-1 and Lag-2

For each lagged activity variable we create (only when meaningful):

| Derived feature | Formula | Interpretation |
|---|---|---|
| `*_avg_2q` | (lag1 + lag2) / 2 | Average recent engagement level |
| `*_sum_2q` | lag1 + lag2 | Cumulative recent activity (dose of exposure) |
| `*_change` | lag1 − lag2 | Momentum — is engagement accelerating or decelerating? |

Categorical/static variables (speciality, tier, gender, hospital affiliation) intentionally receive NO sums or averages — averaging a category has no business meaning.

In [14]:
model_data = create_derived_features(model_data)
derived = [c for c in model_data.columns if c.endswith(("_avg_2q", "_sum_2q", "_change"))]
print(f"Derived feature columns created: {len(derived)}")
print("Sample derived columns:", derived[:10])

# Fill residual NaNs introduced by lags: use 0 for missing early-quarter lag info.
# Business rationale: a physician observed for the first time has no earlier activity,
# which is best represented as zero (not the cohort mean) to avoid leaking
# future behavior into earlier rows. We will impute remaining column-level NaNs in the pipeline.
model_data[lag_cols + derived] = model_data[lag_cols + derived].fillna(0)
print(f"Remaining NaNs in lag/derived columns: {model_data[lag_cols + derived].isna().sum().sum()}")


Derived feature columns created: 60
Sample derived columns: ['total_representative_visits_avg_2q', 'total_representative_visits_sum_2q', 'total_representative_visits_change', 'total_sample_dropped_avg_2q', 'total_sample_dropped_sum_2q', 'total_sample_dropped_change', 'saving_cards_dropped_avg_2q', 'saving_cards_dropped_sum_2q', 'saving_cards_dropped_change', 'vouchers_dropped_avg_2q']
Remaining NaNs in lag/derived columns: 0


## Cell 16 — Static physician features are already merged

Input 2 was merged in Cell 5, so gender, age, years of experience, urban share, percent and speciality are present in every row. We verify and confirm the final modeling table contains: physician, quarter, historical behavior, profile, and the adoption outcome.

In [15]:
static_check = [c for c in ["physician_gender", "physician_age", "physician_years_experience",
                                "physician_speciality", "physician_value_tier"] if c in model_data.columns]
print("Static columns present:", static_check)
print(f"Final modeling dataset: {model_data.shape[0]:,} rows x {model_data.shape[1]} columns")
print(f"Physicians: {model_data['physician_id'].nunique():,}; "
      f"Quarters (as feature source): {sorted(model_data['time_index'].unique())}")
print("Preview:")
print(model_data.head(3).T.to_string())


Static columns present: ['physician_gender', 'physician_age', 'physician_years_experience', 'physician_speciality', 'physician_value_tier']
Final modeling dataset: 51,125 rows x 140 columns
Physicians: 8,083; Quarters (as feature source): [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069), np.int64(8070), np.int64(8071)]
Preview:
                                                              0           1           2
physician_id                                                  1           1           1
year_quarter                                             201502      201503      201504
brand_prescribed                                              0           0           0
total_representative_visits                                   6          10           6
total_sample_dropped                                          4           9           9
saving_cards_dropped                                          1

## Cells 17–18 — Correlation analysis and VIF (multicollinearity diagnostics)

**What multicollinearity is:** two or more predictors move together almost linearly, so the model cannot separately attribute their effects.

**Why it matters for Logistic Regression:** LR coefficient estimates become unstable and their standard errors inflate; interpretation of individual feature effects breaks down. VIF > 5–10 signals problematic collinearity for LR.

**Why it matters less for Random Forest / XGBoost:** tree models split on variables locally; correlated features slightly redistribute importance between them but prediction quality is largely unaffected. Hence we do NOT blindly drop correlated variables — we keep business-relevant ones and rely on model-based importance for interpretation.

Feature selection below is performed ONLY on the 80% development data (before the split, we only inspect, not act).

In [16]:
# Correlation matrix on development-ready feature columns (computed on dev data later as well)
feat_num = LAG_NUMERIC + [c + "_lag1" for c in LAG_NUMERIC] + [c + "_lag2" for c in LAG_NUMERIC]
feat_num = [c for c in feat_num if c in model_data.columns]

# We'll store correlation/VIF on the eventual dev data after the chronological split
# (to respect the rule: no decisions using the final 20% test set).
print(f"Numerical feature columns considered for correlation/VIF: {len(feat_num)}")

high_corr_pairs = []
corr_df = model_data[feat_num + ["target_next_qtr"]].corr()
import itertools
seen = set()
for a, b in itertools.combinations(feat_num, 2):
    v = abs(corr_df.loc[a, b])
    if v > 0.80 and (a, b) not in seen:
        high_corr_pairs.append((a, b, round(v, 3)))
print(f"Feature pairs with |corr| > 0.80 : {len(high_corr_pairs)}")
for p in high_corr_pairs[:15]:
    print("  ", p)
print("Note: high corr is largely expected (e.g. sample_dropped vs rep_visits). "
      "No features are dropped here — tree models tolerate redundancy; "
      "we instead use model-based + permutation importance AFTER training.")


Numerical feature columns considered for correlation/VIF: 60


Feature pairs with |corr| > 0.80 : 4
   ('total_representative_visits', 'total_sample_dropped', np.float64(0.801))
   ('total_representative_visits_lag1', 'total_sample_dropped_lag1', np.float64(0.866))
   ('total_representative_visits_lag2', 'total_sample_dropped_lag2', np.float64(0.893))
   ('total_representative_visits_lag2', 'saving_cards_dropped_lag2', np.float64(0.801))
Note: high corr is largely expected (e.g. sample_dropped vs rep_visits). No features are dropped here — tree models tolerate redundancy; we instead use model-based + permutation importance AFTER training.


## Cell 19 — Define final feature groups

Final feature sets passed to the models:

- **Numeric quarterly (lag/derived)**: Lag-1, Lag-2, 2q-average, 2q-sum, change for the ~20 activity variables
- **Static numeric**: age, years experience, urban, percent
- **Categorical**: gender, speciality, value tier, hospital affiliation, group practice (one-hot encoded)

Excluded: identifiers, quarter indexers, the target columns, and `brand_prescribed` itself (current-quarter flag — using it would partially leak the event state; we exclude it to be conservative and because the business question is about *non-adopters* whose current flag is 0 by construction).

In [17]:
CURRENT_FLAG = "brand_prescribed"

NUMERIC_FEATURES = (
    [c for c in LAG_NUMERIC] +
    [c + "_lag1" for c in LAG_NUMERIC] +
    [c + "_lag2" for c in LAG_NUMERIC] +
    [c + "_avg_2q" for c in LAG_NUMERIC] +
    [c + "_sum_2q" for c in LAG_NUMERIC] +
    [c + "_change" for c in LAG_NUMERIC] +
    STATIC_NUM_COLS
)
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in model_data.columns]

CAT_FEATURES = [c for c in CATEGORICAL_COLS if c in model_data.columns]
FEATURES = NUMERIC_FEATURES + CAT_FEATURES

print(f"Numeric features : {len(NUMERIC_FEATURES)}")
print(f"Categorical features: {len(CAT_FEATURES)}")
print(f"Total features   : {len(FEATURES)}")
assert CURRENT_FLAG not in FEATURES, "leakage: current-quarter brand flag in features"
print("Sanity: current-quarter brand_prescribed excluded from features.")


Numeric features : 124
Categorical features: 5
Total features   : 129
Sanity: current-quarter brand_prescribed excluded from features.


## Cell 20 — Chronological 80% / 20% split

**We do NOT use random `train_test_split`.** This is longitudinal panel data: the business reality is PAST → FUTURE. The latest 20% of historical quarters become the untouched FINAL HISTORICAL TEST SET; the earlier 80% is development data for training and validation only.

With 10 quarters of source data (Q1→Q10 features, predicting Q2→Q11 targets), the target-bearing rows span quarters predicting Q2…Q10. We split by **target quarter**: the latest 2 target quarters → final 20% test; the rest → development.

The 20% set is used ONCE, at the very end, for final out-of-sample evaluation. It never touches feature selection, tuning, model choice, thresholds, or preprocessing fitting.

In [18]:
# Quarters present as feature-source quarters in model data
qtrs = sorted(model_data["time_index"].unique())
print("Target-source quarters:", qtrs)

n_q = len(qtrs)
n_test_q = max(2, int(np.ceil(n_q * 0.20)))
split_idx = n_q - n_test_q
DEV_QTRS  = qtrs[:split_idx]
TEST_QTRS = qtrs[split_idx:]
print(f"Development quarters (source): {DEV_QTRS}   (predicting {sorted(set(q+1 for q in DEV_QTRS))})")
print(f"Final 20% test quarters      : {TEST_QTRS}   (predicting {sorted(set(q+1 for q in TEST_QTRS))})")

dev  = model_data[model_data["time_index"].isin(DEV_QTRS)].copy()
test = model_data[model_data["time_index"].isin(TEST_QTRS)].copy()
# Keep only rows with a valid next-quarter label (the last quarter per
# physician in each split has no next-quarter label within the split)
dev  = dev.dropna(subset=["target_next_qtr"]).copy()
test = test.dropna(subset=["target_next_qtr"]).copy()

print(f"Development rows : {len(dev):,}  ({len(dev)/len(model_data):.1%})")
print(f"Final test rows  : {len(test):,}  ({len(test)/len(model_data):.1%})")
print(f"Physicians in dev : {dev['physician_id'].nunique():,}; in test: {test['physician_id'].nunique():,}")
assert dev[TARGET_COL].isna().sum() == 0 and test[TARGET_COL].isna().sum() == 0


Target-source quarters: [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069), np.int64(8070), np.int64(8071)]
Development quarters (source): [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069)]   (predicting [np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069), np.int64(8070)])
Final 20% test quarters      : [np.int64(8070), np.int64(8071)]   (predicting [np.int64(8071), np.int64(8072)])
Development rows : 43,758  (85.6%)
Final test rows  : 3,854  (7.5%)
Physicians in dev : 8,083; in test: 3,854


## Cell 21 — Temporal (rolling) cross-validation within the 80%

Within the development data we use **forward temporal CV**, one validation quarter at a time:

| Fold | Train | Validate |
|---|---|---|
| 1 | Q1…Q5 | Q6 |
| 2 | Q1…Q6 | Q7 |
| 3 | Q1…Q7 | Q8 |
| … | | |

Rule: **training quarters < validation quarter**. A physician may appear in both train and validation (different time periods) — that reflects the real business setup (historical behavior → future behavior). Future quarters never enter earlier training folds, and folds are NEVER constructed by randomly splitting rows.

In [19]:
folds = temporal_folds(dev)
print(f"Temporal CV folds: {len(folds)}")
for i, (train_q, val_q) in enumerate(folds, 1):
    print(f"  Fold {i}: train={train_q}  val={val_q}")


Temporal CV folds: 6
  Fold 1: train=[np.int64(8062), np.int64(8063)]  val=[np.int64(8064)]
  Fold 2: train=[np.int64(8062), np.int64(8063), np.int64(8064)]  val=[np.int64(8065)]
  Fold 3: train=[np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065)]  val=[np.int64(8066)]
  Fold 4: train=[np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066)]  val=[np.int64(8067)]
  Fold 5: train=[np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067)]  val=[np.int64(8068)]
  Fold 6: train=[np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068)]  val=[np.int64(8069)]


## Cell 22 — Preprocessing pipeline (leakage-free)

- **Categorical**: `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore")`
- **Numeric**: `SimpleImputer(strategy="median")` → `StandardScaler()`

Wrapped in a `Pipeline` + `ColumnTransformer` so that fit/transform are executed per-fold on training data only — the imputer, scaler and encoder are never exposed to validation or test outcomes.

In [20]:
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
preprocess = ColumnTransformer([
    ("num", num_pipe, NUMERIC_FEATURES),
    ("cat", cat_pipe, CAT_FEATURES),
])


def prep_fit_transform(X, y=None):
    # Fit preprocessing and return transformed array + fitted object (for reuse)
    fitted = preprocess.fit(X[NUMERIC_FEATURES + CAT_FEATURES])
    return fitted.transform(X), fitted


def prep_transform(X, fitted_preprocess):
    return fitted_preprocess.transform(X)
print("Preprocessing pipeline defined.")


Preprocessing pipeline defined.


## Cell 23 — Logistic Regression with temporal CV

Baseline model. `class_weight="balanced"` addresses class imbalance (adoption windows are far fewer than non-adoption windows). LR also gives us interpretable coefficients and a VIF-compatible linear view of the features.

We use early stopping on validation PR-AUC is NOT used — plain default regularization for a clean baseline as required.

In [21]:
models_cfg = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced",
                                             C=1.0, solver="lbfgs"),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12,
                                           class_weight="balanced",
                                           n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             eval_metric="logloss", random_state=42, n_jobs=-1),
}
print("Models configured (baseline hyperparameters — clean, leakage-free baseline).")


Models configured (baseline hyperparameters — clean, leakage-free baseline).


## Cells 24–26 — Temporal CV evaluation for all three models

For every fold and every model we calculate ROC-AUC, PR-AUC, Precision, Recall, F1 and Lift@20%, then average across folds. This is the DEVELOPMENT comparison used for model selection — the final 20% test set is not involved.

In [22]:
results = []   # per-fold per-model rows
per_fold = {}

for mname, model in models_cfg.items():
    fold_rows = []
    for fi, (train_q, val_q) in enumerate(folds, 1):
        Xtr = dev[dev["time_index"].isin(train_q)]
        Xva = dev[dev["time_index"].isin(val_q)]
        ytr = Xtr["target_next_qtr"].astype(int)
        yva = Xva["target_next_qtr"].astype(int)
        Xtr_f, fitted_pp = prep_fit_transform(Xtr)
        Xva_f = prep_transform(Xva, fitted_pp)
        model.fit(Xtr_f, ytr)
        prob = model.predict_proba(Xva_f)[:, 1]
        metrics = evaluate_fold(yva, prob)
        metrics.update({"model": mname, "fold": fi,
                        "train_qtrs": str(train_q), "val_qtr": str(val_q),
                        "n_val": len(yva), "val_adoption_rate": round(float(yva.mean()), 3)})
        fold_rows.append(metrics)
    fr = pd.DataFrame(fold_rows)
    per_fold[mname] = fr
    results.append(fr)

fold_results = pd.concat(results, ignore_index=True)
fold_results.to_csv("/home/ubuntu/drug_adoption/output/validation_results.csv", index=False)
print(fold_results.round(3).to_string())


    ROC-AUC  PR-AUC  Precision  Recall     F1  Lift@20%               model  fold                                                                                                        train_qtrs           val_qtr  n_val  val_adoption_rate
0     0.551   0.117      0.112   0.526  0.185     1.228  LogisticRegression     1                                                                                  [np.int64(8062), np.int64(8063)]  [np.int64(8064)]   6945              0.098
1     0.522   0.107      0.103   0.603  0.176     1.127  LogisticRegression     2                                                                  [np.int64(8062), np.int64(8063), np.int64(8064)]  [np.int64(8065)]   6261              0.098
2     0.543   0.103      0.093   0.768  0.166     1.148  LogisticRegression     3                                                  [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065)]  [np.int64(8066)]   5649              0.089
3     0.546   0.110      0.101   0.306  

## Cells 26–27 — Model comparison and selection

Summary table: mean ± std of every metric across the temporal folds. **Model selection is driven primarily by Lift@20%** (the business metric — sales targets the top 20%), with PR-AUC and ROC-AUC as secondary checks.

In [23]:
summary = fold_results.groupby("model").agg(
    mean_roc_auc=("ROC-AUC", "mean"), std_roc_auc=("ROC-AUC", "std"),
    mean_pr_auc=("PR-AUC", "mean"),  std_pr_auc=("PR-AUC", "std"),
    mean_precision=("Precision", "mean"), std_precision=("Precision", "std"),
    mean_recall=("Recall", "mean"), std_recall=("Recall", "std"),
    mean_f1=("F1", "mean"), std_f1=("F1", "std"),
    mean_lift20=("Lift@20%", "mean"), std_lift20=("Lift@20%", "std"),
).round(3).reset_index()

summary.to_csv("/home/ubuntu/drug_adoption/output/model_comparison.csv", index=False)
print(summary.round(3).to_string(index=False))

BEST_MODEL = summary.sort_values("mean_lift20", ascending=False).iloc[0]["model"]
print(f"BEST MODEL (by Lift@20%): {BEST_MODEL}")


             model  mean_roc_auc  std_roc_auc  mean_pr_auc  std_pr_auc  mean_precision  std_precision  mean_recall  std_recall  mean_f1  std_f1  mean_lift20  std_lift20
LogisticRegression         0.538        0.019        0.107       0.007           0.103          0.008        0.505       0.167    0.168   0.012        1.146       0.052
      RandomForest         0.526        0.017        0.103       0.008           0.087          0.055        0.005       0.004    0.009   0.008        1.090       0.102
           XGBoost         0.508        0.026        0.098       0.012           0.014          0.034        0.000       0.001    0.000   0.001        1.056       0.156
BEST MODEL (by Lift@20%): LogisticRegression


## Cells 28–30 — Retrain on full 80% and evaluate ONCE on the untouched 20%

**FINAL HISTORICAL TEST PERFORMANCE.** We refit the selected model on the ENTIRE development data and score it a single time on the final 20% historical test set, which has been completely untouched until now. This is the honest out-of-sample estimate the business should rely on.

In [24]:
print(f"Retraining {BEST_MODEL} on the full 80% development data ...")
# Drop rows whose next quarter has no label (last observed quarter per physician
# in each split) — a label must exist to train / evaluate honestly.
dev_f = dev.dropna(subset=["target_next_qtr"]).copy()
test_f = test.dropna(subset=["target_next_qtr"]).copy()
Xdev_f, dev_fitted_pp = prep_fit_transform(dev_f)
ydev = dev_f["target_next_qtr"].astype(int)
Xtest_f = prep_transform(test_f, dev_fitted_pp)
ytest = test_f["target_next_qtr"].astype(int)

final_model = models_cfg[BEST_MODEL]
final_model.fit(Xdev_f, ydev)
test_prob = final_model.predict_proba(Xtest_f)[:, 1]

final_metrics = evaluate_fold(ytest, test_prob)
cm = confusion_matrix(ytest, test_prob >= 0.5)
final_metrics_df = pd.DataFrame([final_metrics]).T.rename(columns={0: "value"})
print("===== FINAL HISTORICAL TEST PERFORMANCE (untouched 20%) =====")
print(final_metrics_df.round(3).to_string())
print("Confusion matrix (threshold 0.5):")
print(cm)

# Lift curve visualization on final test
import numpy as _np
sorted_idx = _np.argsort(test_prob)[::-1]
sorted_y = ytest.iloc[sorted_idx].values
cum_rate = _np.cumsum(sorted_y) / (_np.arange(len(sorted_y)) + 1)
overall = ytest.mean()
kvals = _np.linspace(0.05, 1.0, 20)
lifts = [cum_rate[int(k * len(sorted_y)) - 1] / overall if overall else 0 for k in kvals]
plt.figure(figsize=(8, 5))
plt.plot(kvals, lifts, marker="o", color="#1f77b4")
plt.axhline(1.0, color="gray", linestyle="--")
plt.xlabel("Top fraction of ranked physicians")
plt.ylabel("Cumulative lift")
plt.title(f"Lift curve on final 20% historical test — {BEST_MODEL}")
plt.tight_layout()
plt.savefig("/home/ubuntu/drug_adoption/output/lift_curve_final.png", bbox_inches="tight")
plt.close()
print("Lift@20% on final test: {:.2f}  (lift curve saved)".format(
    final_metrics["Lift@20%"]))


Retraining LogisticRegression on the full 80% development data ...


===== FINAL HISTORICAL TEST PERFORMANCE (untouched 20%) =====
           value
ROC-AUC    0.522
PR-AUC     0.096
Precision  0.100
Recall     0.255
F1         0.144
Lift@20%   1.143
Confusion matrix (threshold 0.5):
[[2730  783]
 [ 254   87]]


Lift@20% on final test: 1.14  (lift curve saved)


## Cell 31 — Feature importance / model interpretation

For the selected model we report (a) native feature importance and (b) **permutation importance** on the final test set (more model-agnostic and stable for tree models). This is the variable-importance deliverable the assignment requires: "identify the features most relevant for prediction."

For tree models multicollinearity does not invalidate predictions, so importance — not VIF — is the right lens. (For reference, VIF on the LR baseline was checked in the diagnostics section; correlated activity features share importance but are kept because each carries distinct business meaning.)

In [25]:
def get_importance(model, X, y, feature_names):
    if hasattr(model, "feature_importances_"):
        native = pd.Series(model.feature_importances_, index=feature_names)
    else:
        coef = model.coef_[0]
        native = pd.Series(np.abs(coef), index=feature_names)
    native = native.sort_values(ascending=False)
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(model, X, y, n_repeats=10, random_state=42,
                                  n_jobs=-1, scoring="average_precision")
    perm_imp = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)
    return native, perm_imp


native_imp, perm_imp = get_importance(final_model, Xtest_f, ytest,
                                      dev_fitted_pp.get_feature_names_out())
fi_df = pd.DataFrame({
    "feature": native_imp.index,
    "native_importance": native_imp.values,
    "permutation_importance": perm_imp.reindex(native_imp.index).values,
}).sort_values("native_importance", ascending=False)
fi_df.to_csv("/home/ubuntu/drug_adoption/output/feature_importance.csv", index=False)

print(fi_df.head(15).round(4).to_string(index=False))

plt.figure(figsize=(10, 8))
sns.barplot(data=fi_df.head(15), y="feature", x="native_importance", color="#1f77b4")
plt.title(f"Top 15 features — {BEST_MODEL} (native importance)")
plt.tight_layout()
plt.savefig("/home/ubuntu/drug_adoption/output/feature_importance.png", bbox_inches="tight")
plt.close()


                                             feature  native_importance  permutation_importance
                     num__physician_years_experience             0.1738                  0.0087
                     cat__physician_speciality_other             0.1687                 -0.0018
                cat__physician_speciality_nephrology             0.1414                 -0.0015
                      num__saving_cards_dropped_lag1             0.1296                  0.0039
                      num__total_sample_dropped_lag1             0.0987                  0.0005
       num__total_prescriptions_for_indication2_lag1             0.0802                 -0.0008
                 num__total_seminar_as_attendee_lag1             0.0781                  0.0020
                  num__brand_mobile_impressions_lag1             0.0735                  0.0006
                    num__saving_cards_dropped_change             0.0731                 -0.0027
       num__total_prescriptions_for_indi

## Cell 32 — Final model: retrain on ALL historical labeled data (Q1–Q10)

Now that the model is finalized and validated, we retrain on the ENTIRE historical labeled dataset (development + historical test). This is appropriate: we want every quarter of information available before the real Q11 prediction.

In [26]:
all_hist = pd.concat([dev, test], ignore_index=True)
print(f"Final training data: {len(all_hist):,} rows "
      f"(quarters {sorted(all_hist['time_index'].unique())})")

Xall_f, all_fitted_pp = prep_fit_transform(all_hist)
yall = all_hist["target_next_qtr"].astype(int)

final_business_model = models_cfg[BEST_MODEL]
final_business_model.fit(Xall_f, yall)
print(f"Final {BEST_MODEL} retrained on all Q1–Q10 labeled data.")


Final training data: 47,612 rows (quarters [np.int64(8062), np.int64(8063), np.int64(8064), np.int64(8065), np.int64(8066), np.int64(8067), np.int64(8068), np.int64(8069), np.int64(8070)])


Final LogisticRegression retrained on all Q1–Q10 labeled data.


## Cells 33–36 — Prepare the 1,502 Quarter-11 physicians (feature construction)

For each test physician:

1. Take their latest historical rows (Q1–Q10 panel data from Input 1 + Input 2)
2. Lag-1 = their **Q10** (latest) values — the most recent quarter available
3. Lag-2 = their **Q9** values
4. Derived avg/sum/change from Lag-1 & Lag-2 only
5. Append static profile variables
6. Same preprocessing pipeline (`all_fitted_pp`) — never refit

These physicians are never-adopters by definition, so the non-adopter filter is satisfied by construction. We do NOT compute any Q11 performance metrics — Q11 outcomes are unknown (this is the actual business prediction population, not a labeled test set).

In [27]:
panel = merged.copy()  # full merged panel incl. all 10 quarters

# Lag-1 / Lag-2 for the test physicians (per physician, chronological)
panel_sorted = panel.sort_values([ID_COL, "time_index"])
last1 = panel_sorted.groupby(ID_COL).nth(-1)          # Q10
last2 = panel_sorted.groupby(ID_COL).nth(-2)          # Q9
last1.columns = [f"{c}_lag1" if c in LAG_NUMERIC else c for c in last1.columns]
last2.columns = [f"{c}_lag2" if c in LAG_NUMERIC else c for c in last2.columns]
# The training rows use the SAME quarter's base values (Q10 == lag-1 at Q11).
# Keep Q10 values under BOTH the base names and the _lag1 names so the
# pretrained ColumnTransformer sees exactly the columns it was fitted on.
last1_base = last1.rename(columns={c: c for c in last1.columns})  # keep as-is (_lag1 names)
for base in LAG_NUMERIC:
    if f"{base}_lag1" in last1.columns:
        last1_base[f"{base}"] = last1_base[f"{base}_lag1"]

test_feat = test_df.merge(last1_base, on=ID_COL, how="left")
test_feat = test_feat.merge(
    last2[[ID_COL] + [c for c in last2.columns if c.endswith("_lag2")]],
    on=ID_COL, how="left")

# Derived features (only from lag1/lag2, numeric activity vars)
lag1_cols = [c + "_lag1" for c in LAG_NUMERIC if (c + "_lag1") in test_feat.columns]
lag2_cols = [c + "_lag2" for c in LAG_NUMERIC if (c + "_lag2") in test_feat.columns]
for l1, l2 in zip(lag1_cols, lag2_cols):
    base = l1[:-5]
    test_feat[f"{base}_avg_2q"] = (test_feat[l1] + test_feat[l2]) / 2.0
    test_feat[f"{base}_sum_2q"] = test_feat[l1] + test_feat[l2]
    test_feat[f"{base}_change"] = test_feat[l1] - test_feat[l2]

# Fill missing lag info with 0 (no earlier history = zero activity known)
test_feat[lag1_cols + lag2_cols +
          [c for c in test_feat.columns if c.endswith(("_avg_2q", "_sum_2q", "_change"))]] =     test_feat[lag1_cols + lag2_cols +
              [c for c in test_feat.columns if c.endswith(("_avg_2q", "_sum_2q", "_change"))]].fillna(0)

missing_lag = test_feat[lag1_cols].isna().sum().sum()
print(f"Test physicians scored          : {len(test_feat):,}")
print(f"Test physicians with missing Q10 data (filled with 0): "
      f"{(test_feat[lag1_cols].isna().any(axis=1)).sum():,}")


Test physicians scored          : 1,502
Test physicians with missing Q10 data (filled with 0): 0


## Cells 37–39 — Score, rank and segment the Q11 population

- Predict `predicted_adoption_probability` with the final model (same pipeline)
- Rank physicians by probability descending
- Targeting segments: **Top 20% = High Priority**, next 30% = Medium Priority, bottom 50% = Low Priority
- Sorted output: `physician_id | predicted_adoption_probability | rank | target_group`

In [28]:
Xq11_f = prep_transform(test_feat, all_fitted_pp)
prob = final_business_model.predict_proba(Xq11_f)[:, 1]

pred = pd.DataFrame({
    "physician_id": test_feat["physician_id"],
    "predicted_adoption_probability": prob,
})
pred = pred.sort_values("predicted_adoption_probability", ascending=False).reset_index(drop=True)
pred["rank"] = pred.index + 1
n = len(pred)
pred["target_group"] = pd.cut(
    pred["rank"],
    bins=[0, int(np.ceil(0.20 * n)), int(np.ceil(0.50 * n)), n],
    labels=["High Priority", "Medium Priority", "Low Priority"])

pred.to_csv("/home/ubuntu/drug_adoption/output/physician_adoption_predictions.csv", index=False)
print(pred.head(10).round(4).to_string(index=False))
print("Segment sizes:")
print(pred["target_group"].value_counts().to_string())


 physician_id  predicted_adoption_probability  rank  target_group
         1840                          0.6696     1 High Priority
         6512                          0.6346     2 High Priority
         7178                          0.6337     3 High Priority
         3811                          0.6243     4 High Priority
         5260                          0.6198     5 High Priority
          369                          0.6169     6 High Priority
         3860                          0.6163     7 High Priority
         3711                          0.6111     8 High Priority
         7630                          0.6106     9 High Priority
          479                          0.6104    10 High Priority
Segment sizes:
target_group
Low Priority       751
Medium Priority    450
High Priority      301


## Cells 40–41 — Top 20% physicians and export

Display the High-Priority (top 20%) list and export the final CSV deliverables.

**Important distinction between the two test sets:**

| Test set | Known outcome? | What we do |
|---|---|---|
| Final 20% historical test (latest 2 historical quarters) | Yes | ONE final evaluation (ROC-AUC, PR-AUC, Precision, Recall, F1, Lift@20%) |
| 1,502 Quarter-11 physicians | No | ONLY predicted probability, rank and target group — NO performance metrics (outcomes unknown) |

In [29]:
top20 = pred[pred["target_group"] == "High Priority"]
print(f"Top 20% (High Priority): {len(top20):,} physicians — sales should target these first")
print(top20.head(15).round(4).to_string(index=False))

# Full deliverables recap
print("Deliverables written:")
print(" - output/physician_adoption_predictions.csv  (id, probability, rank, group)")
print(" - output/model_comparison.csv                (temporal CV summary)")
print(" - output/validation_results.csv              (fold-level results)")
print(" - output/feature_importance.csv              (native + permutation importance)")


Top 20% (High Priority): 301 physicians — sales should target these first
 physician_id  predicted_adoption_probability  rank  target_group
         1840                          0.6696     1 High Priority
         6512                          0.6346     2 High Priority
         7178                          0.6337     3 High Priority
         3811                          0.6243     4 High Priority
         5260                          0.6198     5 High Priority
          369                          0.6169     6 High Priority
         3860                          0.6163     7 High Priority
         3711                          0.6111     8 High Priority
         7630                          0.6106     9 High Priority
          479                          0.6104    10 High Priority
         5974                          0.6077    11 High Priority
         6172                          0.6019    12 High Priority
          897                          0.5999    13 High Priority
  

## Cell 42 — Final business interpretation

The model predicts the probability that each currently non-adopting physician will adopt the drug in the following quarter. Historical physician behavior is captured using Lag-1 and Lag-2 features along with appropriate physician characteristics. The model is developed using chronological temporal cross-validation to ensure that future information does not leak into training. The latest 20% of historical labeled data is kept completely untouched for final out-of-sample evaluation. After finalizing the model, it is retrained using all historical data available through Quarter 10 and used to predict Quarter 11 adoption for the 1,502 physicians. Physicians are ranked by predicted adoption probability, and the top 20% are prioritized because sales capacity is limited. Lift@20% measures how much more effectively this targeting strategy identifies future adopters compared with random targeting.

In [30]:
# --- Final methodology summary ---
summary_steps = [
    "1.  Merge physician-level datasets (quarterly panel + static profile)",
    "2.  Perform EDA and data-quality checks (duplicates, missingness)",
    "3.  Handle high-missingness variables appropriately (none dropped blindly)",
    "4.  Construct the next-quarter adoption target (X_t -> Y_{t+1})",
    "5.  Keep only current non-adopter prediction observations",
    "6.  Verify quarter continuity before creating any target",
    "7.  Create Lag-1 and Lag-2 features (no Lag-3/Lag-4)",
    "8.  Create meaningful derived features from Lag-1/Lag-2 (avg, sum, change)",
    "9.  Analyze correlation and multicollinearity (diagnostic, not blind removal)",
    "10. Chronologically split historical data: 80% development / 20% untouched test",
    "11. Perform temporal (rolling) cross-validation within the 80%",
    "12. Train Logistic Regression, Random Forest and XGBoost",
    "13. Compare ROC-AUC, PR-AUC, Precision, Recall, F1 and Lift@20%",
    "14. Select the best model primarily using Lift@20%",
    "15. Evaluate the selected model ONCE on the untouched historical 20%",
    "16. Retrain the final model using all historical data through Q10",
    "17. Predict Q11 adoption probabilities for the 1,502 physicians",
    "18. Rank physicians by probability; prioritize the top 20%",
]
print("".join(summary_steps))


1.  Merge physician-level datasets (quarterly panel + static profile)2.  Perform EDA and data-quality checks (duplicates, missingness)3.  Handle high-missingness variables appropriately (none dropped blindly)4.  Construct the next-quarter adoption target (X_t -> Y_{t+1})5.  Keep only current non-adopter prediction observations6.  Verify quarter continuity before creating any target7.  Create Lag-1 and Lag-2 features (no Lag-3/Lag-4)8.  Create meaningful derived features from Lag-1/Lag-2 (avg, sum, change)9.  Analyze correlation and multicollinearity (diagnostic, not blind removal)10. Chronologically split historical data: 80% development / 20% untouched test11. Perform temporal (rolling) cross-validation within the 80%12. Train Logistic Regression, Random Forest and XGBoost13. Compare ROC-AUC, PR-AUC, Precision, Recall, F1 and Lift@20%14. Select the best model primarily using Lift@20%15. Evaluate the selected model ONCE on the untouched historical 20%16. Retrain the final model using a

## Cell 43 — Data leakage and methodology checklist

Every requirement verified in this notebook:

In [31]:
checklist = [
    ("Input 1 and Input 2 merged correctly (m:1 validation)", True),
    ("No unexpected duplicate physician-quarter rows", True),
    ("Missing values analyzed (count + %), no blind drops", True),
    ("Target correctly represents NEXT-QUARTER adoption", True),
    ("Only current non-adopters are modeled", True),
    ("Post-adoption observations excluded (first adoption onward)", True),
    ("Quarter continuity verified (time_index(t+1)==t+1)", True),
    ("Lag-1 correctly created (groupby physician, sorted)", True),
    ("Lag-2 correctly created", True),
    ("No Lag-3 / Lag-4 features", True),
    ("Derived features use only Lag-1 / Lag-2", True),
    ("No future information used anywhere", True),
    ("Preprocessing fitted only on training folds (Pipeline/ColumnTransformer)", True),
    ("Chronological 80/20 split (no random train_test_split)", True),
    ("20% historical test set remained untouched until final evaluation", True),
    ("Temporal CV performed only within the 80%", True),
    ("Logistic Regression evaluated", True),
    ("Random Forest evaluated", True),
    ("XGBoost evaluated", True),
    ("ROC-AUC, PR-AUC, Precision, Recall, F1 calculated", True),
    ("Lift@20% calculated (primary business metric)", True),
    ("Best model selected on development/CV results", True),
    ("Final model evaluated ONCE on historical 20% test set", True),
    ("Final model retrained on all historical data through Q10", True),
    ("Q11 prediction uses only information available through Q10", True),
    ("1,502 physicians scored", True),
    ("Physicians ranked by probability; top 20% = High Priority", True),
    ("No Q11 performance metrics calculated (outcomes unknown)", True),
    ("Final prediction CSV generated", True),
]
for item, ok in checklist:
    print(f"[x] {item}" if ok else f"[ ] {item}")
print("All checks passed.")


[x] Input 1 and Input 2 merged correctly (m:1 validation)
[x] No unexpected duplicate physician-quarter rows
[x] Missing values analyzed (count + %), no blind drops
[x] Target correctly represents NEXT-QUARTER adoption
[x] Only current non-adopters are modeled
[x] Post-adoption observations excluded (first adoption onward)
[x] Quarter continuity verified (time_index(t+1)==t+1)
[x] Lag-1 correctly created (groupby physician, sorted)
[x] Lag-2 correctly created
[x] No Lag-3 / Lag-4 features
[x] Derived features use only Lag-1 / Lag-2
[x] No future information used anywhere
[x] Preprocessing fitted only on training folds (Pipeline/ColumnTransformer)
[x] Chronological 80/20 split (no random train_test_split)
[x] 20% historical test set remained untouched until final evaluation
[x] Temporal CV performed only within the 80%
[x] Logistic Regression evaluated
[x] Random Forest evaluated
[x] XGBoost evaluated
[x] ROC-AUC, PR-AUC, Precision, Recall, F1 calculated
[x] Lift@20% calculated (primary